# UCS420: Cognitive Computing — Assignment 4
## A Cognitive FAQ System Using Pandas

**Name:** Arnav
**Roll Number:** 1024170152

Last two digits of roll number: **5, 2**
- digit 5 → category[5 % 3] = category[2] = **general**
- digit 2 → category[2 % 3] = category[2] = **general**


## Q1: Build Your Personalized Knowledge Base

In [2]:
import pandas as pd

roll_number = "1024170152"

fixed_entries = [
    {"question": "what is the annual fee", "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge", "category": "billing"},
    {"question": "how to reset password", "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login", "category": "account"},
    {"question": "what are your working hours", "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time", "category": "general"},
    {"question": "how can i pay the fee", "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee", "category": "billing"},
]

# Last two digits of roll number 1024170152 -> 5, 2
# digit 5 -> category[5 % 3] = category[2] = "general"
# digit 2 -> category[2 % 3] = category[2] = "general"

personalized_entries = [
    {"question": "how do i check the status of my application",
     "answer": "You can track your application status from the Dashboard > My Applications section.",
     "keywords": "status application track dashboard", "category": "general"},
    {"question": "where can i find the holiday calendar",
     "answer": "The holiday calendar is published under Resources > Academic Calendar.",
     "keywords": "holiday calendar schedule dates", "category": "general"},
]

all_entries = fixed_entries + personalized_entries
faq_df = pd.DataFrame(all_entries)
faq_df


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i check the status of my application,You can track your application status from the...,status application track dashboard,general
5,where can i find the holiday calendar,The holiday calendar is published under Resour...,holiday calendar schedule dates,general


## Q2: Generate and Score a Hypothesis\nScore a query against every FAQ entry and return matches ranked by confidence.

In [3]:
def score_query(query, df):
    """
    Scores each FAQ entry against the query string.
    Score = number of overlapping words between the query and
    (entry's keywords + question), normalized by number of keyword tokens.
    Returns a DataFrame of matching entries (score > 0) sorted by
    confidence in descending order.
    """
    query_words = set(query.lower().split())
    results = []

    for idx, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        searchable = keyword_words | question_words

        overlap = query_words & searchable
        if len(overlap) == 0:
            continue

        score = len(overlap) / len(keyword_words) if keyword_words else 0

        results.append({
            "question": row["question"],
            "answer": row["answer"],
            "category": row["category"],
            "score": round(score, 2)
        })

    result_df = pd.DataFrame(results)
    if not result_df.empty:
        result_df = result_df.sort_values(by="score", ascending=False).reset_index(drop=True)
    return result_df


# Demo
demo_query = "how do i pay the fee"
print(f"Query: '{demo_query}'\n")
score_query(demo_query, faq_df)


Query: 'how do i pay the fee'



,question,answer,category,score
0,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1.25
1,how do i check the status of my application,You can track your application status from the...,general,1.00
2,what is the annual fee,The annual fee is Rs 500.,billing,0.50
3,where can i find the holiday calendar,The holiday calendar is published under Resour...,general,0.50
4,how to reset password,Go to Settings > Reset Password.,account,0.33


## Q3: Retrieve Questions by Category

In [4]:
def same_category(category_name, df):
    """Returns all questions belonging to the given category."""
    return df[df["category"] == category_name]["question"].reset_index(drop=True)


# Using the category of the personalized entries from Q1 ("general", since both
# roll-number digits 5 and 2 map to category[5 % 3] = category[2 % 3] = "general")
general_questions = same_category("general", faq_df)
print("Questions in category 'general':")
print(general_questions)


Questions in category 'general':
0                    what are your working hours
1    how do i check the status of my application
2          where can i find the holiday calendar
Name: question, dtype: str


## Q4: Add a Keyword and Save to CSV\nPick one entry, ask the user for a new keyword, add it, and save the full DataFrame to a CSV file.

In [5]:
# Pick the entry to update (index 0: "what is the annual fee")
entry_index_to_update = 0
print("Selected entry:", faq_df.loc[entry_index_to_update, "question"])
print("Current keywords:", faq_df.loc[entry_index_to_update, "keywords"])

new_keyword = input("Enter a new keyword to add to this entry: ").strip().lower()

faq_df.loc[entry_index_to_update, "keywords"] = faq_df.loc[entry_index_to_update, "keywords"] + " " + new_keyword

csv_filename = f"{roll_number}_faq_data.csv"
faq_df.to_csv(csv_filename, index=False)

print(f"\nUpdated keywords: {faq_df.loc[entry_index_to_update, 'keywords']}")
print(f"Saved full DataFrame to: {csv_filename}")
faq_df


Selected entry: what is the annual fee
Current keywords: fee cost price charge
Enter a new keyword to add to this entry: annual

Updated keywords: fee cost price charge annual
Saved full DataFrame to: 1024170152_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge annual,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i check the status of my application,You can track your application status from the...,status application track dashboard,general
5,where can i find the holiday calendar,The holiday calendar is published under Resour...,holiday calendar schedule dates,general


## Q5: Count FAQ Entries per Category (groupby)

In [6]:
category_counts = faq_df.groupby("category").size().reset_index(name="count")
print(category_counts)


  category  count
0  account      1
1  billing      2
2  general      3


## Q6: Handle Ties in Scoring\nModify the Q2 scoring function so that if multiple entries tie for the highest score, all of them are printed instead of silently picking one.

In [7]:
def score_query_with_ties(query, df, verbose=True):
    """
    Same scoring logic as score_query, but explicitly detects ties for the
    top score. If more than one entry shares the highest score, all tied
    entries are printed/returned instead of silently returning just one.
    """
    query_words = set(query.lower().split())
    results = []

    for idx, row in df.iterrows():
        keyword_words = set(row["keywords"].lower().split())
        question_words = set(row["question"].lower().split())
        searchable = keyword_words | question_words

        overlap = query_words & searchable
        if len(overlap) == 0:
            continue

        score = len(overlap) / len(keyword_words) if keyword_words else 0

        results.append({
            "question": row["question"],
            "answer": row["answer"],
            "category": row["category"],
            "score": round(score, 2)
        })

    result_df = pd.DataFrame(results)
    if result_df.empty:
        if verbose:
            print(f"Query: '{query}' -> No matches found.")
        return result_df

    result_df = result_df.sort_values(by="score", ascending=False).reset_index(drop=True)

    top_score = result_df.iloc[0]["score"]
    tied_df = result_df[result_df["score"] == top_score].reset_index(drop=True)

    if verbose:
        print(f"Query: '{query}'")
        if len(tied_df) > 1:
            print(f"TIE DETECTED: {len(tied_df)} entries tied at top score {top_score}. Showing all of them:\n")
            print(tied_df)
        else:
            print(f"Top match (score {top_score}):\n")
            print(tied_df)
        print()

    return result_df


# Demo 1: a query that produces a tie (matches both "fee" entries)
score_query_with_ties("fee", faq_df)

# Demo 2: a query that does NOT produce a tie
score_query_with_ties("reset my password", faq_df)


Query: 'fee'
Top match (score 0.25):

                question                                      answer category  \
0  how can i pay the fee  You can pay via UPI, card, or net banking.  billing   

   score  
0   0.25  

Query: 'reset my password'
Top match (score 0.67):

                question                            answer category  score
0  how to reset password  Go to Settings > Reset Password.  account   0.67



,question,answer,category,score
0,how to reset password,Go to Settings > Reset Password.,account,0.67
1,how do i check the status of my application,You can track your application status from the...,general,0.25
